# LSTM MIMO + HHO (Return-Based) — Sumber: Stooq

Model dioptimasi Harris Hawks Optimization untuk dataset **Stooq**. Pipeline preprocessing SAMA PERSIS dengan notebook baseline sumber ini (`01_lstm_mimo_baseline_stooq_return.ipynb`), memakai modul shared di `../src/data_utils_return.py` (percentage-return normalization, bukan harga absolut — lihat README, bagian ekstrapolasi harga di luar rentang training).

Fitness HHO memakai RMSE validasi pada SKALA HARGA HASIL REKONSTRUKSI (bukan MSE return ternormalisasi), agar hyperparameter yang dicari benar-benar mengoptimasi kriteria yang dipakai untuk membandingkan model (BAB 3.6.3).

In [ ]:
# --- Setup Colab: clone repo dari GitHub (skip kalau sudah pernah, atau kalau dijalankan lokal & repo sudah ada) ---
import os

REPO_URL = 'https://github.com/mdn1021/lstm-yang-dioptimasi-HHO.git'
BRANCH = 'claude/optimized-model-performance-ipdvtz'
REPO_DIR = 'lstm-yang-dioptimasi-HHO'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL}

%cd {REPO_DIR}/notebooks


In [ ]:
import sys, json, os, time, random, logging
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

from data_utils_return import prepare_source_return, inverse_return_to_price
from model_utils import build_lstm_mimo, decode_hyperparameters, DEFAULT_EPOCHS, DEFAULT_PATIENCE
from hho_utils import make_objective_function, run_hho_search
from evaluation import print_evaluation_report, summary_table

logging.getLogger('tensorflow').setLevel(logging.ERROR)
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

## 1. Load & Preprocess (Return-Based)

In [ ]:
SOURCE = 'stooq'
N_INPUT = 60      # window size (fixed, BUKAN dioptimasi HHO)
N_FORECAST = 5    # forecast horizon (fixed, BUKAN dioptimasi HHO)

ds = prepare_source_return(SOURCE, n_input=N_INPUT, n_forecast=N_FORECAST, data_dir='../data')

print(f"Sumber       : {ds['source']}")
print(f"Data shape   : {ds['data'].shape}")
print(f"Periode      : {ds['data'].index[0].date()} -> {ds['data'].index[-1].date()}")
print(f"X_train      : {ds['X_train'].shape}")
print(f"X_val        : {ds['X_val'].shape}")
print(f"X_test       : {ds['X_test'].shape}")
print(f"Close (train): min={ds['train_df']['Close'].min():.2f}  max={ds['train_df']['Close'].max():.2f}")
print(f"Close (test) : min={ds['test_df']['Close'].min():.2f}  max={ds['test_df']['Close'].max():.2f}")

## 2. Optimasi Hyperparameter dengan HHO (Fitness = RMSE Harga Hasil Rekonstruksi)

Ruang pencarian (6 dimensi): units layer 1 & 2, dropout layer 1 & 2, learning rate,
batch size. Window (n_input) dan horizon (n_forecast) **tidak** termasuk — keduanya
fixed di seluruh model (baseline maupun HHO) untuk menjaga fair comparison.

Fitness setiap kandidat = RMSE validasi pada skala harga USD (setelah `inverse_return_to_price`), BUKAN val_loss return ternormalisasi — supaya HHO mengoptimasi kriteria yang sama dengan yang dipakai membandingkan model.

In [ ]:
lb  = np.array([ 32,   32,  0.1,  0.1,  1e-4,  16])
ub  = np.array([256,  256,  0.5,  0.5,  1e-2, 128])
dim = 6

SEARCH_AGENTS_NO = 5
MAX_ITER         = 10
NUM_RUNS         = 3

objective_function = make_objective_function(
    ds['X_train'], ds['y_train'], ds['X_val'], ds['y_val'],
    n_input=N_INPUT, n_features=5, n_forecast=N_FORECAST,
    reconstruct_price=inverse_return_to_price,
    scaler=ds['scaler'], base_val=ds['base_val'], y_val_abs=ds['y_val_abs']
)

best_hp, best_fitness, best_solution, all_solutions = run_hho_search(
    objective_function, lb, ub, dim, SEARCH_AGENTS_NO, MAX_ITER, NUM_RUNS
)

units1, units2, drop1, drop2, lr, bs = decode_hyperparameters(best_hp)
print("\n" + "="*65)
print(f"  HYPERPARAMETER TERBAIK - STOOQ")
print("="*65)
print(f"  units1={units1}  units2={units2}  dropout1={drop1:.4f}  "
      f"dropout2={drop2:.4f}  lr={lr:.6f}  batch_size={bs}")
print(f"  Best fitness (val RMSE harga, USD) : {best_fitness:.6f}")

## 3. Plot — HHO Convergence Curve

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(best_solution.convergence) + 1), best_solution.convergence,
         marker='o', markersize=4, lw=1.6)
plt.title('HHO Convergence Curve (Stooq, Return-Based)')
plt.xlabel('Iterasi'); plt.ylabel('Best fitness (RMSE harga, USD)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 4. Latih Model Final dengan Hyperparameter Terbaik (Epoch Penuh)

In [ ]:
model = build_lstm_mimo(N_INPUT, n_features=5, n_forecast=N_FORECAST,
                        units1=units1, units2=units2,
                        drop1=drop1, drop2=drop2, learning_rate=lr)
model.summary()

es = EarlyStopping(monitor='val_loss', patience=DEFAULT_PATIENCE,
                   restore_best_weights=True, verbose=0)

history = model.fit(
    ds['X_train'], ds['y_train'],
    validation_data=(ds['X_val'], ds['y_val']),
    epochs=DEFAULT_EPOCHS, batch_size=bs,
    shuffle=False, callbacks=[es], verbose=1
)
print(f"\nTraining selesai pada epoch ke-{len(history.history['loss'])}")

## 5. Prediksi & Evaluasi (RMSE, MAE, MAPE, DA per Horizon)

In [ ]:
train_pred = inverse_return_to_price(ds['scaler'], model.predict(ds['X_train'], verbose=0),
                                      ds['base_train'], N_FORECAST)
val_pred   = inverse_return_to_price(ds['scaler'], model.predict(ds['X_val'], verbose=0),
                                      ds['base_val'], N_FORECAST)
test_pred  = inverse_return_to_price(ds['scaler'], model.predict(ds['X_test'], verbose=0),
                                      ds['base_test'], N_FORECAST)

splits = {
    'TRAIN': (ds['y_train_abs'], train_pred, ds['base_train']),
    'VAL':   (ds['y_val_abs'],   val_pred,   ds['base_val']),
    'TEST':  (ds['y_test_abs'],  test_pred,  ds['base_test']),
}

print_evaluation_report("LSTM MIMO + HHO (Return-Based) - Stooq", N_FORECAST, splits)
summary_table("LSTM MIMO + HHO (Return-Based) - Stooq", N_FORECAST, splits)

## 6. Plot — Training & Validation Loss (Model Final)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss', ls='--')
plt.title('LSTM MIMO + HHO (Stooq, Return-Based) - Training & Validation Loss')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss (skala return)'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Simpan Model & Metadata

In [ ]:
os.makedirs('../models', exist_ok=True)
model.save(f'../models/lstm_mimo_hho_stooq.h5')

metadata = {
    "model_name": "LSTM MIMO + HHO",
    "source": "stooq",
    "n_input": N_INPUT,
    "n_forecast": N_FORECAST,
    "hyperparameters": {
        "units_1": int(units1), "units_2": int(units2),
        "dropout_rate_1": float(drop1), "dropout_rate_2": float(drop2),
        "learning_rate": float(lr), "batch_size": int(bs)
    },
    "hho_search": {
        "search_agents_no": SEARCH_AGENTS_NO, "max_iter": MAX_ITER,
        "num_runs": NUM_RUNS, "best_fitness_rmse_usd": float(best_fitness)
    },
    "normalization": "MinMaxScaler pada percentage change (return) per-fitur, fit pada train only",
    "split": "70/15/15 (walk-forward, time-ordered)"
}
with open(f'../models/lstm_mimo_hho_stooq_meta.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Model & metadata HHO (Stooq, Return-Based) tersimpan di folder models/.")